In [1]:
!pip install DrissionPage

Defaulting to user installation because normal site-packages is not writeable


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
#获取微博帖子
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib import parse

# 设置为自己的cookies
cookies = {
    'SCF': 'AjZO-ID3_8H-NA7AKqCjQwxbSbuFxsGboG-yO4inlljauPFePBAdq3MJkcedeJ9tfm2ixNxEdfh4Zl4MOTDTGcw.',
    'SINAGLOBAL': '7810380168450.5205.1745129382496',
    'SUBP': '0033WrSXqPxfM725Ws9jqgMF55529P9D9WWy7xJ2lacgPjv-HWvRJFzk5JpX5KMhUgL.FoqEeo.cS0BXeK22dJLoI7LIxHile0BR1K5t',
    'ALF': '1749976341',
    'SUB': '_2A25FIohFDeRhGeBM6VsX9yrIyj2IHXVmQYWNrDV8PUJbkNAYLUnEkW1NROZNSk7nF9sWOL29FV_54Ri7SREgQ29z',
    '_s_tentry': 'weibo.com',
    'Apache': '3681093303505.2896.1747386995002',
    'ULV': '1747386995054:6:3:2:3681093303505.2896.1747386995002:1747114080134',
}


def get_the_list_response(q='话题', the_type='综合', p='页码', timescope="2023-01-01-0:2023-03-01-16"):
    """
    q表示的是话题，type表示的是类别，有：综合，实时，热门，高级，p表示的页码，timescope表示高级的时间，不用高级无需带入
    """
    type_params_url = {
        '综合': [{"q": q, "Refer": "weibo_weibo", "page": p, }, 'https://s.weibo.com/weibo'],
        '实时': [{"q": q, "rd": "realtime", "tw": "realtime", "Refer": "realtime_realtime", "page": p, },
                 'https://s.weibo.com/realtime'],
        '热门': [{"q": q, "xsort": "hot", "suball": "1", "tw": "hotweibo", "Refer": "realtime_hot", "page": p},
                 'https://s.weibo.com/hot'],
        # 高级中的xsort删除后就是普通的排序
        '高级': [{"q": q, "xsort": "hot", "suball": "1", "timescope": f"custom:{timescope}", "Refer": "g", "page": p},
                 'https://s.weibo.com/weibo']
    }

    params, url = type_params_url[the_type]

    headers = {
        'authority': 's.weibo.com',
        'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
        'accept-language': 'zh-CN,zh;q=0.9,en;q=0.8,en-GB;q=0.7,en-US;q=0.6',
        'referer': url + "?" + parse.urlencode(params).replace(f'&page={params["page"]}',
                                                               f'&page={int(params["page"]) - 1}'),
        'sec-ch-ua': '"Chromium";v="116", "Not)A;Brand";v="24", "Microsoft Edge";v="116"',
        'sec-ch-ua-mobile': '?0',
        'sec-ch-ua-platform': '"Windows"',
        'sec-fetch-dest': 'document',
        'sec-fetch-mode': 'navigate',
        'sec-fetch-site': 'same-origin',
        'sec-fetch-user': '?1',
        'upgrade-insecure-requests': '1',
        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36 Edg/116.0.1938.69',
    }
    response = requests.get(url, params=params, cookies=cookies, headers=headers)
    return response


def parse_the_list(text):
    """该函数就是解析网页主题内容的"""
    soup = BeautifulSoup(text, 'html.parser')
    divs = soup.select('div[action-type="feed_list_item"]')
    lst = []
    for div in divs:
        name = div.find('a', class_='name').text
        mid = div.get('mid')
        uid = div.select('div.card-feed > div.avator > a')
        if uid:
            uid = uid[0].get('href').replace('.com/', '?').split('?')[1]
        else:
            uid = None
        time = div.select('div.card-feed > div.content > div.from > a:first-of-type')
        if time:
            time = time[0].string.strip()
        else:
            time = None
        p = div.select('div.card-feed > div.content > p:last-of-type')
        if p:
            p = p[0].strings
            content = '\n'.join([para.replace('\u200b', '').strip() for para in list(p)]).replace('收起\nd', '').strip()
        else:
            content = None
        star = div.select('div.card-act > ul > li:nth-child(3) > a > button > span.woo-like-count')
        if star:
            star = list(star[0].strings)[0]
        else:
            star = None
        comments = div.select('div.card-act > ul > li:nth-child(2) > a')
        if comments:
            comments = list(comments[0].strings)[0]
        else:
            comments = None
        retweets = div.select('div.card-act > ul > li:nth-child(1) > a')
        if retweets:
            retweets = list(retweets[0].strings)[1]
        else:
            retweets = None
        lst.append((name, mid, uid, content, retweets, comments, star, time))
        # with open('评论.txt', 'a', encoding='utf-8') as fp:
        #     fp.write(content)
        #     fp.write('\n')
    df = pd.DataFrame(lst, columns=['name', 'mid', 'uid', 'content', 'retweets', 'comments', 'star', 'time'])
    return df


def get_the_list(q, the_type, p, ):
    df_list = []
    for i in range(1, p + 1):
        response = get_the_list_response(q=q, the_type=the_type, p=i)
        if response.status_code == 200:
            df = parse_the_list(response.text)
            df_list.append(df)
            print(f'第{i}页解析成功！', flush=True)

    return df_list


if __name__ == '__main__':
    # 先设置cookie，换成自己的；
    # the_type 有 综合，实时，热门，高级 具体介绍看get_the_list_response函数
    the_type = '综合'
    q = '小米SU7燃爆事件'
    p = 30
    df_list = get_the_list(q, the_type, p)
    df = pd.concat(df_list)
    df.to_csv('C:\\Users\\hp\\Desktop\\小米su7燃爆.csv', index=False, encoding='utf-8-sig')

In [ ]:
#获取微博评论
import time
from DrissionPage.common import Actions
from DrissionPage import ChromiumPage
from datetime import datetime
import csv

f=open('C:\\Users\\hp\\Desktop\\微博评论(1).csv','a',encoding='utf-8-sig',newline='')
writer=csv.writer(f)
writer.writerow(['用户名', '评论内容', 'ip', '点赞数', '时间', '性别'])
page = ChromiumPage()  # 打开浏览器
ac = Actions(page)
page.listen.start('https://weibo.com/ajax/statuses/buildComments?')
page.get('https://weibo.com/6389872027/Pptdnm4GY')
for j in range(15):  # 控制下拉条下拉
    page.actions.scroll(10000, 0)
    time.sleep(1)
for x in range(1000):
    res = page.listen.wait()  # 等待数据包监听
    data = res.response.body
    comment = data['data']
    for i in comment:
        name = i['user']['screen_name']
        text = i['text_raw']
        try:
            source = i['source']
        except:
            source = '未知'
        like = i['like_counts']
        date = i['created_at']
        dt_obj = datetime.strptime(date, "%a %b %d %H:%M:%S +0800 %Y")
        time = dt_obj.strftime("%Y-%m-%d %H:%M:%S")
        gender = i['user']['gender']
        if gender == 'f':
            sex = '女'
        else:
            sex = '男'
        print(name, text, source, like, time, sex)
        writer.writerow([name, text, source, like, time, sex])

In [ ]:
#这是BERT分词的试用
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')
print(tokenizer.tokenize("小米SU7太帅了"))

In [7]:
import pandas as pd
import jieba
import re
import chardet
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# ✅ 从文件加载中文停用词
with open('C:\\Users\\hp\\Desktop\\baidu_stopwords.txt', encoding='utf-8') as f:
    chinese_stopwords = set([line.strip() for line in f])

# 合并中英文停用词
stopwords = chinese_stopwords.union(ENGLISH_STOP_WORDS)

# 自动检测文件编码
def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read(100000))  # 读取前100KB以检测编码
    print(f"检测到文件编码：{result['encoding']}")
    return result['encoding']

# 文本清洗函数
def clean_text(text):
    if pd.isnull(text):
        return ''
    
    # 去除换行符、HTML标签、表情、非中英文字符等
    text = re.sub(r'<[^>]+>', '', text)  # 去HTML标签
    text = re.sub(r'[^\u4e00-\u9fa5a-zA-Z0-9]', ' ', text)  # 保留中英文和数字
    text = re.sub(r'\s+', ' ', text).strip()  # 去多余空格

    # 分词
    words = jieba.lcut(text)

    # 去除停用词
    filtered_words = [w for w in words if w.lower() not in stopwords and len(w.strip()) > 1]

    return ' '.join(filtered_words)

# 主处理函数
def process_csv(input_file, output_file):
    # 手动使用 open 读取并处理编码问题
    with open(input_file, 'r', encoding='gb18030', errors='replace') as f:
        df = pd.read_csv(f)

    print("🚿 正在清洗文本内容……")
    df['cleaned_content'] = df['content'].astype(str).apply(clean_text)

    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"✅ 清洗完成，结果已保存到：{output_file}")

# 🚀 执行处理
process_csv('C:\\Users\\hp\\Desktop\\小米su7燃爆.csv', 'C:\\Users\\hp\\Desktop\\微博帖子.csv')

🚿 正在清洗文本内容……
✅ 清洗完成，结果已保存到：C:\Users\hp\Desktop\微博帖子.csv


In [ ]:
#删去重复分词
import pandas as pd
import chardet

def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read(100000))
    print(f"检测到文件编码：{result['encoding']}")
    return result['encoding']

file_path = 'C:\\Users\\hp\\Desktop\\微博帖子.csv'
encoding = detect_encoding(file_path)

try:
    df = pd.read_csv(file_path, encoding=encoding)
except UnicodeDecodeError:
    print(f"用 {encoding} 读取失败，尝试用 gb18030 编码")
    df = pd.read_csv(file_path, encoding='gb18030')

def remove_duplicates(text):
    words = text.split()
    seen = set()
    unique_words = []
    for w in words:
        if w not in seen:
            seen.add(w)
            unique_words.append(w)
    return ' '.join(unique_words)

df['cleaned_content'] = df['cleaned_content'].astype(str).apply(remove_duplicates)
df.to_csv('C:\\Users\\hp\\Desktop\\微博帖子new.csv', index=False, encoding='utf-8')

In [13]:
import pandas as pd
import jieba
import re
import chardet
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF, LatentDirichletAllocation
from gensim.models import Word2Vec
import numpy as np
from transformers import BertTokenizer

# 中文停用词表+ 英文停用词
def load_chinese_stopwords():
    stopwords = set()
    with open('C:\\Users\\hp\\Desktop\\baidu_stopwords.txt', encoding='utf-8') as f:
        for line in f:
            stopwords.add(line.strip())
    return stopwords

# 自动检测文件编码
def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read(100000))  # 读取前100KB以检测编码
    print(f"检测到文件编码：{result['encoding']}")
    return result['encoding']

# 文本清洗函数
def clean_text(text):
    if pd.isnull(text):
        return ''
    
    # 去除换行符、HTML标签、表情、非中英文字符等
    text = re.sub(r'<[^>]+>', '', text)  # 去HTML标签
    text = re.sub(r'[^\u4e00-\u9fa5a-zA-Z0-9]', ' ', text)  # 保留中英文和数字
    text = re.sub(r'\s+', ' ', text).strip()  # 去多余空格

    # 分词
    words = jieba.lcut(text)

    # 去除停用词
    filtered_words = [w for w in words if w.lower() not in stopwords and len(w.strip()) > 1]
    return ' '.join(filtered_words)

# 文本向量化函数
def vectorize_text(text):
    words = multi_model_tokenize(text)
    filtered_words = [w for w in words if w.lower() not in stopwords and len(w.strip()) > 1]
    
    # 主题模型（示例：NMF）
    nmf = NMF(n_components=5, random_state=1).fit(bag_of_words.values())
    topic_vector = nmf.transform([bag_of_words])[0]
    
    return vector, topic_vector

# 主处理函数
def process_csv(input_file, output_file):
    # 手动使用 open 读取并处理编码问题
    with open(input_file, 'r', encoding='gb18030', errors='replace') as f:
        df = pd.read_csv(f)

    print("🚿 正在清洗文本内容……")
    df['cleaned_content'] = df['cleaned_content'].astype(str).apply(clean_text)

    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"✅ 清洗完成，结果已保存到：{output_file}")

process_csv('C:\\Users\\hp\\Desktop\\微博帖子new.csv','C:\\Users\\hp\\Desktop\\微博帖子.csv')

🚿 正在清洗文本内容……
✅ 清洗完成，结果已保存到：C:\Users\hp\Desktop\微博帖子.csv


In [ ]:
import pandas as pd
import jieba
import re
import chardet
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF, LatentDirichletAllocation
from gensim.models import Word2Vec
import numpy as np
from transformers import BertTokenizer

# 中文停用词表+ 英文停用词
def load_chinese_stopwords():
    stopwords = set()
    with open('C:\\Users\\hp\\Desktop\\baidu_stopwords.txt', encoding='utf-8') as f:
        for line in f:
            stopwords.add(line.strip())
    return stopwords

# 自动检测文件编码
def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read(100000))  # 读取前100KB以检测编码
    print(f"检测到文件编码：{result['encoding']}")
    return result['encoding']

# 文本清洗函数
def clean_text(text):
    if pd.isnull(text):
        return ''
    
    # 去除换行符、HTML标签、表情、非中英文字符等
    text = re.sub(r'<[^>]+>', '', text)  # 去HTML标签
    text = re.sub(r'[^\u4e00-\u9fa5a-zA-Z0-9]', ' ', text)  # 保留中英文和数字
    text = re.sub(r'\s+', ' ', text).strip()  # 去多余空格

    # 分词
    words = jieba.lcut(text)

    # 去除停用词
    filtered_words = [w for w in words if w.lower() not in stopwords and len(w.strip()) > 1]
    return ' '.join(filtered_words)

# 文本向量化函数
def vectorize_text(text):
    words = multi_model_tokenize(text)
    filtered_words = [w for w in words if w.lower() not in stopwords and len(w.strip()) > 1]
    
    # 主题模型（示例：NMF）
    nmf = NMF(n_components=5, random_state=1).fit(bag_of_words.values())
    topic_vector = nmf.transform([bag_of_words])[0]
    
    return vector, topic_vector

# 主处理函数
def process_csv(input_file, output_file):
    # 手动使用 open 读取并处理编码问题
    with open(input_file, 'r', encoding='gb18030', errors='replace') as f:
        df = pd.read_csv(f)

    print("🚿 正在清洗文本内容……")
    df['cleaned_content'] = df['cleaned_content'].astype(str).apply(clean_text)

    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"✅ 清洗完成，结果已保存到：{output_file}")

process_csv("C:\\Users\\hp\\Desktop\\微博评论(1).csv","C:\\Users\\hp\\Desktop\\微博评论.csv")